In [ ]:
#0. 작업 준비
import numpy as np
import torch
import matplotlib.pyplot as plt

from torch.utils import data
from torchvision import datasets,transforms
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from transformers.convert_slow_tokenizer import import_protobuf

USE_MPS = torch.backends.mps.is_available()
DEVICE=torch.device('mps' if USE_MPS else 'cpu')

# DCGAN 모델 생성

In [ ]:
BATCH_SIZE = 128
EPOCHS = 30

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,)) # [-1, 1]
])
tr_ds = datasets.MNIST(root='../../data', train=True, transform=transform, download=True)
tr_ds_loader = data.DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [ ]:
Z_DIM = 3
IMG_C = 1
IMG_SIZE = 28

In [ ]:
D = nn.Sequential(
    nn.Conv2d(IMG_C, 64, 4, 2, 1, bias=False), #[64, 14, 14]
    nn.LeakyReLU(0.2, inplace=True),
    nn.Conv2d(64, 128, 4, 2, 1, bias=False), #[128, 7, 7]
    nn.BatchNorm2d(128),
    nn.LeakyReLU(0.2, inplace=True),
    nn.Flatten(),
    nn.Linear(128 * 7 * 7, 1),
    nn.Sigmoid()
)

In [ ]:
G = nn.Sequential(
    nn.Linear(Z_DIM, 128*7*7),
    nn.BatchNorm1d(128*7*7),
    nn.ReLU(),
    nn.Unflatten(1, (128, 7, 7)), #[128, 7, 7]
    nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False), #[64, 14, 14]
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.ConvTranspose2d(64, IMG_C, 4, 2, 1, bias=False), #[1, 28, 28]
    nn.Tanh()
)

In [ ]:
G = G.to(DEVICE)
D = D.to(DEVICE)

In [ ]:
criterion = nn.BCELoss()
d_opt = optim.Adam(D.parameters(), lr=0.0002)
g_opt = optim.Adam(G.parameters(), lr=0.0002)

In [ ]:
z_noise = torch.randn(BATCH_SIZE, Z_DIM, device=DEVICE)
for epoch in range(EPOCHS):
    for i, (x, _) in enumerate(tr_ds_loader):
        data = x.to(DEVICE)
        # data.size[0] 배치 사이즈
        r_label = torch.ones(BATCH_SIZE, 1, device=DEVICE)
        f_label = torch.zeros(BATCH_SIZE, 1, device=DEVICE)

        z = torch.randn(BATCH_SIZE, Z_DIM, device=DEVICE)
        f_img = G(z)

        real_out = D(data)
        d_loss_real = criterion(real_out, r_label)
        real_sc = real_out.detach()

        fake_out = D(f_img.detach())
        d_loss_fake = criterion(fake_out, f_label)
        fake_sc = fake_out.detach()

        d_loss = d_loss_real + d_loss_fake

        g_opt.zero_grad()
        d_opt.zero_grad()
        d_loss.backward()
        d_opt.step()

        fake_out_ck_g = D(f_img)
        g_loss = criterion(fake_out_ck_g, r_label)

        g_opt.zero_grad()
        d_opt.zero_grad()
        g_loss.backward()
        g_opt.step()
    print(f'{epoch+1} 회, d_loss: {d_loss.item()}, g_loss: {g_loss.item()} D(G(z)): {fake_sc.mean().item()} D(x): {real_sc.mean().item()}')

    with torch.no_grad():
        G.eval()
        v_img = G(z_noise) #(64, 1, 28, 28)
        v_img = (v_img+1)/2

        f, a = plt.subplots(8, 8, figsize=(8, 8))
        for i, ax in enumerate(a.flat):
            img = v_img[i].cpu().permute(1, 2, 0).numpy()
            ax.imshow(img.squeeze(), cmap='gray')
            ax.axis('off')
        plt.tight_layout()
        plt.show()
        G.train()

# 패션 데이터를 이용하여 DCGAN을 구현하시오

In [ ]:
from torch.utils import data
from torchvision import datasets,transforms

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,)) # [-1, 1]
])
tr_ds = datasets.FashionMNIST(root='../../data', train=True, transform=transform, download=True)
tr_ds_loader = data.DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [ ]:
Z_DIM = 300
IMG_C = 1
IMG_SIZE = 28

In [ ]:
D = nn.Sequential(
    nn.Conv2d(IMG_C, 64, 4, 2, 1, bias=False),  #[64, 14, 14]
    nn.LeakyReLU(0.2, inplace=True),
    nn.Conv2d(64, 128, 4, 2, 1, bias=False),  #[128, 7, 7]
    nn.BatchNorm2d(128),
    nn.LeakyReLU(0.2, inplace=True),
    nn.Flatten(),
    nn.Linear(128 * 7 * 7, 1),
    nn.Sigmoid()
)
G = nn.Sequential(
    nn.Linear(Z_DIM, 128 * 7 * 7),
    nn.BatchNorm1d(128 * 7 * 7),
    nn.ReLU(),
    nn.Unflatten(1, (128, 7, 7)),  #[128, 7, 7]
    nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),  #[64, 14, 14]
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.ConvTranspose2d(64, IMG_C, 4, 2, 1, bias=False),  #[1, 28, 28]
    nn.Tanh()
)

In [ ]:
G = G.to(DEVICE)
D = D.to(DEVICE)
criterion = nn.BCELoss()
d_opt = optim.Adam(D.parameters(), lr=0.0002)
g_opt = optim.Adam(G.parameters(), lr=0.0002)

In [ ]:
z_noise = torch.randn(BATCH_SIZE, Z_DIM, device=DEVICE)
for epoch in range(EPOCHS):
    for i, (x, _) in enumerate(tr_ds_loader):
        data = x.to(DEVICE)
        # data.size[0] 배치 사이즈
        r_label = torch.ones(BATCH_SIZE, 1, device=DEVICE)
        f_label = torch.zeros(BATCH_SIZE, 1, device=DEVICE)

        z = torch.randn(BATCH_SIZE, Z_DIM, device=DEVICE)
        f_img = G(z)

        real_out = D(data)
        d_loss_real = criterion(real_out, r_label)
        real_sc = real_out.detach()

        fake_out = D(f_img.detach())
        d_loss_fake = criterion(fake_out, f_label)
        fake_sc = fake_out.detach()

        d_loss = d_loss_real + d_loss_fake

        g_opt.zero_grad()
        d_opt.zero_grad()
        d_loss.backward()
        d_opt.step()

        fake_out_ck_g = D(f_img)
        g_loss = criterion(fake_out_ck_g, r_label)

        g_opt.zero_grad()
        d_opt.zero_grad()
        g_loss.backward()
        g_opt.step()
    print(
        f'{epoch + 1} 회, d_loss: {d_loss.item()}, g_loss: {g_loss.item()} D(G(z)): {fake_sc.mean().item()} D(x): {real_sc.mean().item()}')

    with torch.no_grad():
        G.eval()
        v_img = G(z_noise)  #(64, 1, 28, 28)
        v_img = (v_img + 1) / 2

        f, a = plt.subplots(8, 8, figsize=(8, 8))
        for i, ax in enumerate(a.flat):
            img = v_img[i].cpu().permute(1, 2, 0).numpy()
            ax.imshow(img.squeeze(), cmap='gray')
            ax.axis('off')
        plt.tight_layout()
        plt.show()
        G.train()

# 데이터 로드

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

IMG_SIZE = 128
BATCH_SIZE = 32
transform = transforms.Compose([
    # 데이터 증강
    # 이미지 크기 resize
    transforms.Resize(IMG_SIZE),
    # 텐서화
    transforms.ToTensor(),
    # 스케일 정리
    # 이미지 net 의 데이터 기본 처리 방식
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.334, 0.225))
])
tr_ds = datasets.ImageFolder(root='../../data/dogs-vs-cats_data/train', transform=transform)
tt_ds = datasets.ImageFolder(root='../../data/dogs-vs-cats_data/test', transform=transform)
val_ds = datasets.ImageFolder(root='../../data/dogs-vs-cats_data/val', transform=transform)

tr_ds_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
tt_ds_loader = DataLoader(tt_ds, batch_size=BATCH_SIZE, shuffle=True)
val_ds_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
x, y = next(iter(tr_ds_loader))

In [ ]:
import os
import shutil
import pathlib
old_dir=pathlib.Path('../../data/dogs-vs-cats_data/train')
new_dir=pathlib.Path('../../data/dogs-vs-cats_data')
def make_subset(sub_n,s_idx,e_idx):
    for i in ['cat','dog']:
        dir=new_dir/sub_n/i
        os.makedirs(dir)
        f_ns=[f'{i}.{n}.jpg' for n in range(s_idx,e_idx)]
        for f_n in f_ns:
            shutil.copyfile(src=old_dir/f_n,dst=dir/f_n)
make_subset('train',0,1000)
make_subset('val',1000,2000)
make_subset('test',2000,2500)

In [ ]:
# 대표적인 비표준
# 1. 한 폴더에 모든 클래스의 파일이 있는
# 2. 라벨이 csv파일로 정리 후 따로 저장
# 3. json 정보와 파일 정보 혼합

In [ ]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class F_dataset(Dataset):
    def __init__(self, path, transform):
        self.img_dir = path
        self.img_f = list(self.img_dir.glob('*.jpg'))
        if len(self.img_f) == 0:
            print('로드할 파일이 없습니다')
        self.transform = transform

    def __len__(self):
        return len(self.img_f)
    def __getitem__(self, idx):
        img_path = self.img_f[idx]

        image = Image.open(img_path).convert('RGB')

        # 내용 변경 파악
        label_name = img_path.stem.split('.')[0].lower() #cat.0.jpg
        if label_name == 'cat':
            label = 0
        elif label_name == 'dog':
            label = 1
        else:
            raise ValueError(f'파일 클래스 내용이 없는 파일이 로드되었습니다 {img_path.name}')

        if self.transform:
            image = self.transform(image)
        return image, label

base_dir = Path('../../data/dogs-vs-cats/train')
transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
])
tr_ds = F_dataset(base_dir, transform)
tr_ds_loader = DataLoader(tr_ds, batch_size=4, shuffle=True)

In [ ]:
x, y = next(iter(tr_ds_loader))

In [ ]:
x.shape

In [ ]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms


class F_dataset(Dataset):
    def __init__(self, path, transform):
        self.img_dir = path
        self.img_f = list(self.img_dir.glob('*.jpg'))
        if len(self.img_f) == 0:
            print('로드할 파일이 없습니다')
        self.transform = transform

    def __len__(self):
        return len(self.img_f)
    def __getitem__(self, idx):
        img_path = self.img_f[idx]

        image = Image.open(img_path).convert('RGB')

        # 내용 변경 파악
        label_name = img_path.stem.split('.')[0].lower() #cat.0.jpg
        if label_name == 'cat':
            label = 0
        elif label_name == 'dog':
            label = 1
        else:
            raise ValueError(f'파일 클래스 내용이 없는 파일이 로드되었습니다 {img_path.name}')

        if self.transform:
            image = self.transform(image)
        return image, label

base_dir = Path('../../data/dogs-vs-cats/train')
transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
])
all_ds = F_dataset(base_dir, transform)
all_ds_n = len(all_ds)

# 일반 권장
tr_ds_n = int(all_ds_n*.7)
tt_ds_n = int(all_ds_n*.3)
val_ds_n = int(tr_ds_n*.2)
tr_ds_n = int(tr_ds_n*.8)
print(all_ds_n,tr_ds_n, tt_ds_n, val_ds_n, tr_ds_n+tt_ds_n+val_ds_n)
tr_ds, tt_ds, val_ds = random_split(all_ds, [tr_ds_n, tt_ds_n, val_ds_n])
print(tr_ds_n, tt_ds_n, val_ds_n, len(tr_ds), len(tt_ds), len(val_ds))

tr_ds_loader = DataLoader(tr_ds, batch_size=4, shuffle=True)
tt_ds_loader = DataLoader(tt_ds, batch_size=4, shuffle=True)
val_ds_loader = DataLoader(val_ds, batch_size=4, shuffle=True)

# 디렉토리 파일을 로드 하여, class 를 활용한 DNN, CNN 모델을 구축하고 모델을 학습/추론하는 코드를 완성하시오
1. dogs-vs-cats 데이터셋을 활용하여 완성하시오
2. 데이터는 최대한 모든 데이터를 이용하시오
3. 학습, 추론은 함수로 정리하시오

## DNN


In [1]:
# import 함수
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

USE_MPS = torch.backends.mps.is_available()
DEVICE = torch.device("mps" if USE_MPS else "cpu")

In [2]:
# 데이터 셋 나누기 (train, test, validation)
class F_dataset(Dataset):
    def __init__(self, path, transform):
        self.img_dir = path
        self.img_f = list(self.img_dir.glob('*.jpg'))
        if len(self.img_f) == 0:
            print('로드할 파일이 없습니다')
        self.transform = transform

    def __len__(self):
        return len(self.img_f)
    def __getitem__(self, idx):
        img_path = self.img_f[idx]

        image = Image.open(img_path).convert('RGB')

        # 내용 변경 파악
        label_name = img_path.stem.split('.')[0].lower() #cat.0.jpg
        if label_name == 'cat':
            label = 0
        elif label_name == 'dog':
            label = 1
        else:
            raise ValueError(f'파일 클래스 내용이 없는 파일이 로드되었습니다 {img_path.name}')

        if self.transform:
            image = self.transform(image)
        return image, label

base_dir = Path('../../data/dogs-vs-cats/train')
transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])
all_ds = F_dataset(base_dir, transform)
all_ds_n = len(all_ds)

# 일반 권장
tr_ds_n = int(all_ds_n*.7)
tt_ds_n = int(all_ds_n*.3)
val_ds_n = int(tr_ds_n*.2)
tr_ds_n = int(tr_ds_n*.8)
print(all_ds_n,tr_ds_n, tt_ds_n, val_ds_n, tr_ds_n+tt_ds_n+val_ds_n)
tr_ds, tt_ds, val_ds = random_split(all_ds, [tr_ds_n, tt_ds_n, val_ds_n])
print(tr_ds_n, tt_ds_n, val_ds_n, len(tr_ds), len(tt_ds), len(val_ds))

tr_ds_loader = DataLoader(tr_ds, batch_size=128, shuffle=True, drop_last=True)
tt_ds_loader = DataLoader(tt_ds, batch_size=128, shuffle=True, drop_last=True)
val_ds_loader = DataLoader(val_ds, batch_size=128, shuffle=True, drop_last=True)

25000 14000 7500 3500 25000
14000 7500 3500 14000 7500 3500


In [7]:
# 모델 정의
class DNN_Model(nn.Module):
    def __init__(self, input_n, hidden_ns, output_n, dropout_p=0.2):
        super().__init__()
        self.fc_in = nn.Linear(input_n, hidden_ns[0])
        self.fc_h_l = nn.ModuleList([nn.Linear(hidden_ns[i], hidden_ns[i+1]) for i in range(len(hidden_ns)-1)])
        self.fc_out = nn.Linear(hidden_ns[-1], output_n)

        self.dropout_p = dropout_p
        self.input_n = input_n
        self.hidden_ns = hidden_ns
        self.output_n = output_n

    def forward(self, x):
        # 벡터화
        x = x.view(-1, self.input_n)
        # 입력계층 연산
        x = F.relu(self.fc_in(x))
        x = F.dropout(x, training=self.training, p=self.dropout_p)
        # 은닉계층 연산
        for i in range(len(self.fc_h_l)):
            x = F.relu(self.fc_h_l[i](x))
            x = F.dropout(x, training=self.training, p=self.dropout_p)
        # 출력계층 연산
        out = self.fc_out(x)
        return out

In [8]:
# 모델 학습
INPUT_SIZE = 3*100*100
model = DNN_Model(INPUT_SIZE, [256, 128, 68], 1).to(DEVICE)
opt = optim.Adam(model.parameters(), lr=0.001)


def train(model, tr_ds_loader, opt):
    model.train()
    for i, (x, y) in enumerate(tr_ds_loader):
        data, target = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        py = model(data)
        target_float = target.float().unsqueeze(1)
        loss = F.binary_cross_entropy_with_logits(py, target_float)
        loss.backward()
        opt.step()


def evaluate(model, tt_ds_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in tt_ds_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            py = model(data)
            target_float = target.float().unsqueeze(1)
            test_loss += F.binary_cross_entropy_with_logits(py, target_float, reduction='sum').item()
            pred = (torch.sigmoid(py) > 0.5).float()
            correct += pred.eq(target_float).sum().item()
    test_loss /= len(tt_ds_loader.dataset)
    test_accuracy = correct / len(tt_ds_loader.dataset) * 100.
    return test_loss, test_accuracy


EPOCHS = 30
for i in range(1, EPOCHS + 1):
    train(model, tr_ds_loader, opt)
    train_loss, train_accuracy = evaluate(model, tr_ds_loader)
    val_loss, val_accuracy = evaluate(model, val_ds_loader)

    print(f'Epoch: {i:2d}/{EPOCHS} | Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')

print('-' * 30)
print("최종 모델 성능 평가 (Test Set)")
test_loss, test_accuracy = evaluate(model, tt_ds_loader)
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%')

Epoch:  1/30 | Train Loss: 0.6439, Train Acc: 62.61% | Val Loss: 0.6497, Val Acc: 59.46%
Epoch:  2/30 | Train Loss: 0.6172, Train Acc: 64.86% | Val Loss: 0.6363, Val Acc: 61.74%
Epoch:  3/30 | Train Loss: 0.6099, Train Acc: 66.57% | Val Loss: 0.6354, Val Acc: 60.74%
Epoch:  4/30 | Train Loss: 0.5792, Train Acc: 69.46% | Val Loss: 0.6228, Val Acc: 63.14%
Epoch:  5/30 | Train Loss: 0.5715, Train Acc: 70.68% | Val Loss: 0.6292, Val Acc: 62.14%
Epoch:  6/30 | Train Loss: 0.5415, Train Acc: 72.16% | Val Loss: 0.6207, Val Acc: 63.14%
Epoch:  7/30 | Train Loss: 0.5165, Train Acc: 74.32% | Val Loss: 0.6247, Val Acc: 64.20%
Epoch:  8/30 | Train Loss: 0.5037, Train Acc: 74.75% | Val Loss: 0.6269, Val Acc: 64.03%
Epoch:  9/30 | Train Loss: 0.5036, Train Acc: 75.67% | Val Loss: 0.6293, Val Acc: 63.00%
Epoch: 10/30 | Train Loss: 0.4412, Train Acc: 79.09% | Val Loss: 0.6474, Val Acc: 64.26%
Epoch: 11/30 | Train Loss: 0.4106, Train Acc: 81.71% | Val Loss: 0.6523, Val Acc: 63.94%
Epoch: 12/30 | Train 

# CNN

In [3]:
#CNN
class CNN_Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(32, 512)
        self.fc2 = nn.Linear(512, 1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        out = self.fc2(x)
        return out

In [ ]:
model = CNN_Model().to(DEVICE)
opt = optim.Adam(model.parameters(), lr=0.001)


def train(model, tr_ds_loader, opt):
    model.train()
    for i, (x, y) in enumerate(tr_ds_loader):
        data, target = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        target_float = target.float()
        py = model(data).squeeze(1)
        loss = F.binary_cross_entropy_with_logits(py, target_float)
        loss.backward()
        opt.step()

def evaluate(model, data_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            py = model(data).squeeze(1)
            target_float = target.float()
            test_loss += F.binary_cross_entropy_with_logits(py, target_float, reduction='sum').item()
            pred = (py > 0).float()
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(data_loader.dataset)
    test_accuracy = correct / len(data_loader.dataset) * 100.
    return test_loss, test_accuracy

# 학습 루프
EPOCHS = 30
for i in range(1, EPOCHS + 1):
    train(model, tr_ds_loader, opt)
    # 검증 데이터로 성능 확인
    train_loss, train_accuracy = evaluate(model, tr_ds_loader)
    val_loss, val_accuracy = evaluate(model, val_ds_loader)
    print(f'Epoch: {i:2d}/{EPOCHS} | Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')

# 최종 테스트
test_loss, test_accuracy = evaluate(model, tt_ds_loader)
print('-' * 30)
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%')